# Lab: Fine-Tune a Small LLM with LoRA

**Goal:** Fine-tune a small instruction-following language model so it responds like a concise, practical business-school AI assistant.

This codebook uses:

- Hugging Face `transformers`
- Hugging Face `datasets`
- Hugging Face `peft` for LoRA
- Optional MLflow logging

Recommended compute: **Databricks Runtime 17.3 LTS ML**. CPU execution is feasible for this simple lab, but GPU execution is recommended for a real fine-tuning task.


## 0. Environment setup



In [0]:
# install all required libraries
%pip install -U "transformers<5.0.0,>=4.41.0" datasets accelerate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 129.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 19.0.1
    Not uninstalling pyarrow at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-b94b5f1b-a122-4a06-8d43-1fc3d1233334
    Can't uninstall 'pyarrow'. No files were found to uninstall.
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.30.2
    Not uninstallin

After the install cell, restart Python once so the installed package versions are cleanly loaded. Then continue with the next cells.

In [0]:
# Restart Python after notebook-scoped package installation.

try:
    dbutils.library.restartPython()
except NameError:
    print("Not running inside Databricks, skipping dbutils.library.restartPython().")


## 1. Imports and path configuration

This notebook supports two storage options:

- **Workspace/Repo local files:** keep the `data/` folder beside the notebook. This is easiest for teaching.
- **Unity Catalog Volume:** set `USE_UC_VOLUME = True` and update `UC_VOLUME_BASE`.

The notebook saves the LoRA adapter under the selected working directory.

In [0]:
import os
from pathlib import Path
import json
import torch

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, PeftModel

# Set this to True if you prefer to stores data and outputs in a Unity Catalog Volume.
USE_UC_VOLUME = False

# Replace shanz3 with your MUID
UC_VOLUME_BASE = "/Volumes/isa632_7474656346303369/default/isa632_boopatt/llm_finetuning_tutorial"

if USE_UC_VOLUME:
    BASE_DIR = Path(UC_VOLUME_BASE)
else:
    # Works when the data folder is stored beside the notebook in a Databricks Repo/Workspace file directory.
    BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs" / "qwen_lora_business_assistant"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


2026-05-06 16:26:47.998427: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-06 16:26:48.007051: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-06 16:26:48.065162: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778084808.154646    8645 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778084808.173399    8645 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778084808.197947    8645 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

BASE_DIR: /Workspace/Users/boopatt@miamioh.edu/Module10 (1)/isa632-m10_lab6
DATA_DIR: /Workspace/Users/boopatt@miamioh.edu/Module10 (1)/isa632-m10_lab6/data
OUTPUT_DIR: /Workspace/Users/boopatt@miamioh.edu/Module10 (1)/isa632-m10_lab6/outputs/qwen_lora_business_assistant
CUDA available: False


## 2. Create the tiny dataset inside Databricks

Run this cell only if the `data/` files are not already available beside the notebook. It writes the same small JSONL dataset used in the original tutorial.

In [0]:
# Run this cell only if DATA_DIR does not already contain train.jsonl and validation.jsonl.
# It creates the toy dataset directly inside Databricks.

CREATE_DATA_IF_MISSING = True

train_examples = [{'id': 1, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A customer says: 'My grocery bill keeps rising every week. What can I do?'", 'output': 'Provide a practical, empathetic answer about budgeting, comparing unit prices, planning meals, using store brands, and tracking recurring purchases.'}, {'id': 2, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A student asks: 'What is generative AI in simple terms?'", 'output': 'Explain that generative AI creates new content such as text, images, code, or summaries by learning patterns from large datasets.'}, {'id': 3, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A manager asks: 'How can AI help with inventory planning?'", 'output': 'Explain that AI can forecast demand, detect stockout risks, optimize reorder timing, and support better supplier decisions.'}, {'id': 4, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A customer says: 'The chatbot gave me the wrong refund policy.'", 'output': 'Apologize, suggest checking the official policy, explain that AI can make mistakes, and offer to connect the customer with a human agent.'}, {'id': 5, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A student asks: 'What does fine-tuning mean?'", 'output': 'Explain that fine-tuning adapts a pretrained model to a specific task or style using a smaller, focused dataset.'}, {'id': 6, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A professor asks: 'Why use LoRA instead of full fine-tuning?'", 'output': 'Explain that LoRA updates a small set of adapter weights, reducing memory, compute, and storage while preserving most model knowledge.'}, {'id': 7, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A business analyst asks: 'Can AI predict food prices?'", 'output': 'Explain that AI can combine historical prices, economic indicators, and local signals, but predictions should be evaluated carefully.'}, {'id': 8, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A student asks: 'What is a foundation model?'", 'output': 'Explain that a foundation model is a large pretrained model that can be adapted to many downstream tasks.'}, {'id': 9, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A customer says: 'I need help writing a professional email.'", 'output': 'Ask for the purpose, recipient, tone, and key points, then offer to draft a concise message.'}, {'id': 10, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A manager asks: 'What are risks of using AI in business?'", 'output': 'Mention inaccurate outputs, bias, privacy risks, security concerns, overreliance, and the need for human review.'}, {'id': 11, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A student asks: 'What is prompt engineering?'", 'output': 'Explain that prompt engineering means writing clear instructions and context so an AI model produces better outputs.'}, {'id': 12, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A user says: 'Summarize this report but keep the citations.'", 'output': 'Explain that the summary should preserve key claims, keep citation markers attached to supported statements, and avoid adding unsupported claims.'}, {'id': 13, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A customer says: 'The AI response sounded rude.'", 'output': 'Apologize, acknowledge the tone issue, restate the answer politely, and offer to improve the wording.'}, {'id': 14, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A student asks: 'What is training data?'", 'output': 'Explain that training data consists of examples used to teach a model patterns, relationships, and desired outputs.'}, {'id': 15, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A manager asks: 'Should we replace employees with AI?'", 'output': 'Recommend using AI to augment work, automate routine tasks, and keep humans involved in judgment, empathy, and accountability.'}, {'id': 16, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A student asks: 'What is overfitting?'", 'output': 'Explain that overfitting happens when a model memorizes training examples and performs poorly on new data.'}, {'id': 17, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A professor asks: 'How should students document AI use?'", 'output': 'Recommend stating the tool used, purpose, prompts or workflow, human edits, and how accuracy was checked.'}, {'id': 18, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A user asks: 'Can AI cite sources it never read?'", 'output': 'Explain that it should not; citations should only support claims actually derived from the cited source.'}, {'id': 19, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A manager asks: 'How can we evaluate an AI chatbot?'", 'output': 'Suggest measuring accuracy, helpfulness, safety, response time, user satisfaction, escalation rate, and error patterns.'}, {'id': 20, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A student asks: 'What is an embedding?'", 'output': 'Explain that an embedding is a numeric representation of text, images, or other data that captures meaning for search or prediction.'}, {'id': 21, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A customer asks: 'Why did my coupon not apply?'", 'output': 'Explain possible reasons such as expiration, product exclusions, minimum purchase requirements, or account restrictions, and suggest checking the coupon terms.'}, {'id': 22, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A student asks: 'What is retrieval augmented generation?'", 'output': "Explain that RAG retrieves relevant documents first, then uses them as context for the model's answer."}, {'id': 23, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A manager asks: 'What is a good first AI project?'", 'output': 'Recommend a narrow, high-value, low-risk process such as summarizing support tickets, drafting routine emails, or classifying requests.'}, {'id': 24, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A customer says: 'I want cheaper meal ideas for my family.'", 'output': 'Offer simple strategies such as batch cooking, low-cost proteins, seasonal produce, frozen vegetables, and planning around weekly sales.'}, {'id': 25, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A student asks: 'What is model inference?'", 'output': 'Explain that inference is using a trained model to generate predictions or responses for new inputs.'}, {'id': 26, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A business owner asks: 'How can AI improve customer service?'", 'output': 'Explain that AI can answer common questions, summarize conversations, route issues, draft replies, and provide 24/7 first-line support.'}, {'id': 27, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A student asks: 'What is a tokenizer?'", 'output': 'Explain that a tokenizer breaks text into smaller units called tokens that the model can process.'}, {'id': 28, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A professor asks: 'How do I make an AI assignment fair?'", 'output': 'Suggest clear rules, disclosure requirements, task-specific rubrics, process evidence, and assessments that require human judgment.'}, {'id': 29, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A user asks: 'Write a short LinkedIn post about our AI workshop.'", 'output': 'Draft a concise, positive post mentioning the workshop topic, audience, key takeaway, and appreciation for participants.'}, {'id': 30, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A manager asks: 'Can we trust AI-generated analysis?'", 'output': 'Explain that AI output should be verified with data, domain expertise, and clear evaluation because models can be confidently wrong.'}, {'id': 31, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A student asks: 'What is supervised learning?'", 'output': 'Explain that supervised learning trains a model using examples that include both inputs and correct outputs or labels.'}, {'id': 32, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A customer asks: 'Can you explain this bill?'", 'output': 'Ask for the bill details, then explain charges clearly, separate fixed fees from usage-based costs, and flag anything unusual.'}, {'id': 33, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A manager asks: 'What makes an AI use case valuable?'", 'output': 'Explain that a valuable use case has clear business impact, available data, measurable outcomes, manageable risk, and user adoption.'}, {'id': 34, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A student asks: 'What is zero-shot learning?'", 'output': 'Explain that zero-shot learning means asking a model to perform a task without task-specific training examples.'}]

validation_examples = [{'id': 35, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A user asks: 'Make this text less formal.'", 'output': 'Rewrite the text in a simpler, friendlier tone while preserving the original meaning.'}, {'id': 36, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A professor asks: 'What is a hallucination in AI?'", 'output': 'Explain that a hallucination is when a model produces information that sounds plausible but is incorrect or unsupported.'}, {'id': 37, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A manager asks: 'How do we protect sensitive data when using AI?'", 'output': 'Suggest minimizing data sharing, removing personal information, using approved tools, setting access controls, and reviewing vendor policies.'}, {'id': 38, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A student asks: 'What is validation data?'", 'output': 'Explain that validation data is held-out data used to check model performance during development and tune settings.'}, {'id': 39, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A customer asks: 'Can you compare two products for me?'", 'output': 'Ask for the product names and comparison criteria, then compare features, price, use cases, and tradeoffs.'}, {'id': 40, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A user asks: 'Generate quiz questions from this chapter.'", 'output': 'Create clear questions covering key concepts, mix question types, and include answers if requested.'}]

if CREATE_DATA_IF_MISSING:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    train_path = DATA_DIR / "train.jsonl"
    validation_path = DATA_DIR / "validation.jsonl"

    if not train_path.exists():
        with open(train_path, "w", encoding="utf-8") as f:
            for row in train_examples:
                f.write(json.dumps(row) + "\n")
        print("Wrote", train_path)
    else:
        print("Found existing", train_path)

    if not validation_path.exists():
        with open(validation_path, "w", encoding="utf-8") as f:
            for row in validation_examples:
                f.write(json.dumps(row) + "\n")
        print("Wrote", validation_path)
    else:
        print("Found existing", validation_path)


Wrote /Workspace/Users/boopatt@miamioh.edu/Module10 (1)/isa632-m10_lab6/data/train.jsonl
Wrote /Workspace/Users/boopatt@miamioh.edu/Module10 (1)/isa632-m10_lab6/data/validation.jsonl


## 3. Load and inspect the dataset

Each row has three fields:

- `instruction`: the role or behavior we want the model to follow.
- `input`: the user request.
- `output`: the target response.


In [0]:
dataset = load_dataset(
    "json",
    data_files={
        "train": str(DATA_DIR / "train.jsonl"),
        "validation": str(DATA_DIR / "validation.jsonl"),
    }
)

print(dataset)
print(dataset["train"][0])


/databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:56: UserWarning: The cache_dir for this dataset is /tmp/.hf.data.cache, which is not a persistent path.Therefore, if/when the cluster restarts, the downloaded dataset will be lost.The persistent storage options for this workspace/cluster config are: [UC Volumes].Please update either `cache_dir` or the environment variable `HF_DATASETS_CACHE`to be under one of the following root directories: ['/Volumes/']
  warnings.warn(warning_message)
/databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:24: UserWarning: During large dataset downloads, there could be multiple progress bar widgets that can cause performance issues for your notebook or browser. To avoid these issues, use `datasets.utils.logging.disable_progress_bar()` to turn off the progress bars.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'instruction', 'input', 'output'],
        num_rows: 34
    })
    validation: Dataset({
        features: ['id', 'instruction', 'input', 'output'],
        num_rows: 6
    })
})
{'id': 1, 'instruction': 'Respond as a helpful business-school AI assistant. Be concise, accurate, and practical.', 'input': "A customer says: 'My grocery bill keeps rising every week. What can I do?'", 'output': 'Provide a practical, empathetic answer about budgeting, comparing unit prices, planning meals, using store brands, and tracking recurring purchases.'}


## 4. Choose a small foundation model

For this lab, we use `Qwen/Qwen2.5-0.5B-Instruct`, a small open instruction model. The goal is not to create the best possible assistant, but to show the fine-tuning workflow clearly.


In [0]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

model.config.use_cache = False
print("Loaded model:", MODEL_NAME)


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

## 5. Format examples as chat-style prompts

We create a prompt from `instruction` + `input`, then train the model to produce `output`. The loss is applied only to the assistant response, not the user prompt.

In [0]:
MAX_LENGTH = 512

def build_messages(example, include_output=True):
    messages = [
        {"role": "system", "content": example["instruction"]},
        {"role": "user", "content": example["input"]},
    ]
    if include_output:
        messages.append({"role": "assistant", "content": example["output"]})
    return messages

def tokenize_example(example):
    prompt_text = tokenizer.apply_chat_template(
        build_messages(example, include_output=False),
        tokenize=False,
        add_generation_prompt=True,
    )
    full_text = tokenizer.apply_chat_template(
        build_messages(example, include_output=True),
        tokenize=False,
        add_generation_prompt=False,
    )

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    labels = full["input_ids"].copy()
    prompt_len = min(len(prompt_ids), len(labels))
    labels[:prompt_len] = [-100] * prompt_len

    full["labels"] = labels
    return full

tokenized = dataset.map(tokenize_example, remove_columns=dataset["train"].column_names)
print(tokenized)
print("Tokenized sample keys:", tokenized["train"][0].keys())


## 6. Add LoRA adapters

LoRA freezes the original model weights and trains a small number of adapter parameters. This makes the lab faster and cheaper than full fine-tuning.

In [0]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 7. Train

This uses a tiny number of steps so the lesson finishes quickly. For a real project, increase the data size, epochs, and evaluation rigor.

In [0]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_steps=2,
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    report_to=[],  # Change to ["mlflow"] if you want MLflow autologging.
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
)

trainer.train()


## 8. Save the LoRA adapter

The saved adapter is small compared with the full foundation model. You can copy this folder to a shared location or register/log it as part of a broader MLflow workflow.

In [0]:
ADAPTER_DIR = OUTPUT_DIR / "final_lora_adapter"
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("Saved adapter to:", ADAPTER_DIR)

## 9. Test the fine-tuned adapter

This reloads the base model and attaches the fine-tuned LoRA adapter.

In [0]:
def generate_answer(model, tokenizer, user_question, system_instruction="You are a concise business-school AI assistant."):
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=140,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

test_question = "Explain why hallucination is a risk when using generative AI for business decisions."
print(generate_answer(model, tokenizer, test_question))


## 10. Compare base model vs. fine-tuned adapter

For a stronger lesson, ask students to compare the base model and fine-tuned model on the same prompts. With this tiny dataset, changes may be modest, but students should see how the workflow works.

In [0]:
# Reload base model only.
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
base_model.config.use_cache = False

print("BASE MODEL RESPONSE")
print(generate_answer(base_model, tokenizer, test_question))

print("\nFINE-TUNED LORA RESPONSE")
print(generate_answer(model, tokenizer, test_question))